# Delta UniForm to ADW External Table Demo

This notebook creates a Delta table with UniForm enabled for Iceberg, writes data to it, and then calls an ADW stored procedure that mounts the latest Iceberg-compatible metadata file as an ADW external table.

Run `refresh_iceberg_external_table.sql` in ADW before executing this notebook.

## Demo configuration

For a cleaner demo, keep all user-editable values in one place. The notebook passes only the base Delta/UniForm table location to ADW; the ADW procedure resolves the latest `vN.metadata.json` under `metadata/`.

In [ ]:
demo = {
    # Spark table name used for the Delta + UniForm table.
    "spark_table_name": "default.default.uniform_table_01",
    # ADW external table name to create/recreate.
    "adw_table_name": "UNIFORM_TABLE_01",
    # ADW procedure created from refresh_iceberg_external_table.sql.
    "adw_procedure_name": "REFRESH_ICEBERG_EXTERNAL_TABLE",
    # DBMS_CLOUD credential already created in ADW.
    "adw_credential_name": "...",
    # OCI region corresponding to the Object Storage bucket.
    "object_storage_region": "us-ashburn-1",
}

table_name = demo["spark_table_name"]

## Create or replace the Delta table with UniForm enabled

The table can be partitioned as well. ADW still reads the table through Iceberg metadata and does not require a non-partitioned layout.

If you want to demonstrate partitioning, replace the `CREATE TABLE` statement with a `PARTITIONED BY (...)` variant.

In [ ]:
# Minimal UniForm-enabled Delta table.
# In a more complete setup, also verify any required UniForm properties for your runtime.
spark.sql(f"""
  CREATE TABLE IF NOT EXISTS {table_name}(id INT)
  USING DELTA
  TBLPROPERTIES('delta.universalFormat.enabledFormats' = 'iceberg')
""")

## Write sample data

A Delta commit with UniForm enabled should generate or update Iceberg-compatible metadata under the table's `metadata/` directory.

In [ ]:
spark.sql(f"INSERT INTO {table_name} VALUES (1)")

## Inspect the table location

The ADW procedure expects the base table location in `oci://bucket@namespace/path` form, not the specific metadata file.

In [ ]:
table_location = spark.sql(f"DESCRIBE DETAIL {table_name}").head().location
table_location

## Connect to ADW with python-oracledb

This demo assumes **mTLS** is required on the ADW side.

Nuances:
- If the ADB page shows **Mutual TLS**, you need the wallet.
- For `python-oracledb` "thin" mode, the wallet directory typically needs `tnsnames.ora` and `ewallet.pem`.
- `config_dir` points to `tnsnames.ora` and `wallet_location` points to `ewallet.pem`. In practice they are often the same directory.
- `wallet_password` is the password used when the wallet zip was downloaded, not the database user password.

In [ ]:
# python-oracledb must be available on the compute image or environment.
import oracledb

adw = {
    # Schema in ADW where the refresh procedure was created.
    "user": "...",
    # Database password for the ADW schema.
    "password": "...",
    # TNS alias from tnsnames.ora, for example amitaidpadw_medium.
    "dsn": "...",
    # Wallet directory used for mTLS. This should contain tnsnames.ora and ewallet.pem.
    "config_dir": "...",
    "wallet_location": "...",
    # Password used when downloading the wallet zip from OCI.
    "wallet_password": "...",
}

conn = oracledb.connect(
    user=adw["user"],
    password=adw["password"],
    dsn=adw["dsn"],
    config_dir=adw["config_dir"],
    wallet_location=adw["wallet_location"],
    wallet_password=adw["wallet_password"],
)

## Call the ADW procedure

The procedure receives:
- the ADW external table name
- the ADW `DBMS_CLOUD` credential name
- the OCI region
- the base Delta/UniForm table location in OCI URI form

Inside ADW, the procedure resolves the latest `metadata/vN.metadata.json` and recreates the external table against that latest metadata file.

In [ ]:
try:
    cur = conn.cursor()
    cur.callproc(
        demo["adw_procedure_name"],
        [
            demo["adw_table_name"],
            demo["adw_credential_name"],
            demo["object_storage_region"],
            table_location,
        ],
    )
    conn.commit()
finally:
    cur.close()
    conn.close()

## What to verify next

In ADW, verify that:
- the external table was recreated successfully
- querying the ADW external table returns the rows written through Spark
- additional writes to the Delta table create newer `vN.metadata.json` files that can be picked up by rerunning the procedure